<a href="https://colab.research.google.com/github/flaviafguerreiro-debug/Desafios_/blob/main/Desafio1_API_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
# Instala as bibliotecas usadas no guia.
!pip install -q gspread faker

In [22]:
# Importa as ferramentas para autenticação, geração de dados e sorteios.
import json, gspread
from google.colab import userdata
from faker import Faker
import random

# Lê a credencial salva nos Secrets e conecta à planilha.
info = json.loads(userdata.get("GCP_SA_JSON"))
gc = gspread.service_account_from_dict(info)
# Exibe o e-mail da conta autenticada.
print("Conectado como:", info["client_email"])

Conectado como: desafio1-python@project-ffeef7bb-38ac-43d7-86a.iam.gserviceaccount.com


In [23]:
# Substitua pelos nomes EXATOS de coluna da ficha do seu tema
# Define as colunas que formarão o cabeçalho da tabela.
COLUNAS = ["cliente_id", "seguimento", "canal_preferido" , "data_cadastro"]

# Escreve o cabeçalho na primeira célula e confere o resultado.
ws.update(range_name="A1", values=[COLUNAS])
print("Cabeçalho confere:", ws.row_values(1) == COLUNAS)

Cabeçalho confere: True


In [27]:
# Configura os geradores aleatórios para produzir resultados reproduzíveis.
fake = Faker("pt_BR")
Faker.seed(42)
random.seed(42)

def gerar_linha(i):
    # Cria um registro fictício usando regras de dados condizentes com os nomes das colunas
    return {
        "cliente_id": i,
        "seguimento": random.choice(["Bronze", "Prata", "Ouro", "Premium"]),
        "canal_preferido": random.choice(["WhatsApp", "E-mail", "Telefone", "Aplicativo"]),
        "data_cadastro": fake.date_this_decade().isoformat(),
    }

# Gera 10 registros numerados de 1 a 10.
linhas = [gerar_linha(i) for i in range(1, 11)]

# Organiza os valores na ordem do cabeçalho; os nomes de colunas agora coincidem perfeitamente.
valores = [[linha[c] for c in COLUNAS] for linha in linhas]
# Acrescenta os registros à planilha e informa quantos foram enviados.
ws.append_rows(valores, value_input_option="RAW")
print(len(valores), "linhas enviadas")

10 linhas enviadas


In [25]:
# Lê todos os valores da aba para conferir o conteúdo enviado.
valores_lidos = ws.get_all_values()
# Exibe a quantidade de linhas, confere o cabeçalho e mostra exemplos.
print("Linhas (cabeçalho + dados):", len(valores_lidos))    # esperado: 11
print("Cabeçalho confere:", valores_lidos[0] == COLUNAS)
print(valores_lidos[:3])

Linhas (cabeçalho + dados): 31
Cabeçalho confere: True
[['cliente_id', 'seguimento', 'canal_preferido', 'data_cadastro'], ['1', '2824', 'PK-1', 'Brenda Alves'], ['2', '1409', 'PK-2', 'Sra. Isabelly Câmara']]
